In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/landing/EnterpriseCustomer360_POC_Complex.xml"
)

In [0]:
# Databricks notebook source
# IDMC Enterprise Customer 360 - Migration runner
# Process ALL mappings and generate a consolidated Excel report.
# Dry-run by default. Generated PySpark is NOT executed.

import json
import os
import sys
import traceback
import glob
from datetime import datetime, timezone


# ============================================================
# 1. PROJECT AND INPUT CONFIGURATION
# ============================================================

PROJECT_ROOT = (
    "/Workspace/Users/"
    "lokeswaraseetaramasastry.k@tcs.com/"
    "IDMC_Databricks_LLM"
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# EXACT XML PATH CONFIRMED BY YOU
XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/landing/EnterpriseCustomer360_POC_Corrected.xml"
)

DEFAULT_OUTPUT_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark/"
    "enterprise_customer360"
)

LLM_MODEL = "databricks-gpt-oss-120b"


# ============================================================
# 2. NOTEBOOK WIDGETS
# ============================================================

def ensure_text_widget(name, default_value):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default_value)


ensure_text_widget("output_dir", DEFAULT_OUTPUT_DIR)
ensure_text_widget("dry_run", "true")
ensure_text_widget("execution_approved", "false")
ensure_text_widget("allow_synthetic_merge_keys", "false")

OUTPUT_DIR = dbutils.widgets.get("output_dir").strip()

DRY_RUN = (
    dbutils.widgets.get("dry_run").strip().lower() == "true"
)

EXECUTION_APPROVED = (
    dbutils.widgets.get("execution_approved").strip().lower() == "true"
)

ALLOW_SYNTHETIC_MERGE_KEYS = (
    dbutils.widgets.get("allow_synthetic_merge_keys")
    .strip()
    .lower() == "true"
)

# None means ALL mappings from the XML.
SELECTED_MAPPING_NAME = None

REVIEW_DIR = os.path.join(OUTPUT_DIR, "human_reviews")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REVIEW_DIR, exist_ok=True)


# ============================================================
# 3. SAFETY VALIDATIONS
# ============================================================

if not OUTPUT_DIR:
    raise ValueError("output_dir cannot be empty.")

if not os.path.exists(XML_PATH):
    raise FileNotFoundError(
        f"XML file does not exist: {XML_PATH}"
    )

if not DRY_RUN and not EXECUTION_APPROVED:
    raise RuntimeError(
        "Live execution is blocked. "
        "Explicit approval is required."
    )

if DRY_RUN and EXECUTION_APPROVED:
    raise ValueError(
        "execution_approved must be false in dry-run mode."
    )

if not DRY_RUN and ALLOW_SYNTHETIC_MERGE_KEYS:
    raise RuntimeError(
        "Live execution with synthetic merge keys is blocked."
    )


# ============================================================
# 4. DATABRICKS CONTEXT
# ============================================================

context = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
)

WORKSPACE_URL = context.apiUrl().get()
DATABRICKS_TOKEN = context.apiToken().get()

if not WORKSPACE_URL:
    raise RuntimeError(
        "Databricks workspace URL is unavailable."
    )

if not DATABRICKS_TOKEN:
    raise RuntimeError(
        "Databricks notebook token is unavailable."
    )


print("=" * 90)
print("IDMC ENTERPRISE CUSTOMER 360 MIGRATION RUNNER")
print("=" * 90)
print("XML:", XML_PATH)
print("Mapping selection: ALL")
print("LLM:", LLM_MODEL)
print("Dry run:", DRY_RUN)
print("Execution approved:", EXECUTION_APPROVED)
print("Output:", OUTPUT_DIR)
print("=" * 90)


# ============================================================
# 5. IMPORT MIGRATION PIPELINE
# ============================================================

from pipeline.migration_pipeline import MigrationPipeline


# ============================================================
# 6. RUN ALL MAPPINGS
# ============================================================

run_started_at = datetime.now(timezone.utc)

results = []
metadata_snapshot = {}
pipeline_error = None
pipeline_result = None

print("\nStarting migration pipeline for ALL mappings...")

try:
    pipeline = MigrationPipeline(
        xml_path=XML_PATH,
        spark=spark,
        output_dir=OUTPUT_DIR,
        workspace_url=WORKSPACE_URL,
        databricks_token=DATABRICKS_TOKEN,
        llm_model=LLM_MODEL,
        selected_mapping_name=None,
        dry_run=DRY_RUN,
        execution_approved=EXECUTION_APPROVED,
        allow_synthetic_merge_keys=ALLOW_SYNTHETIC_MERGE_KEYS,
        verbose=True,
        human_review_dir=REVIEW_DIR,
        approved_review_id=None,
    )

    pipeline_result = pipeline.run()

    metadata_snapshot = (
        getattr(pipeline, "metadata_bundle", {}) or {}
    )

    if isinstance(pipeline_result, list):
        results.extend(pipeline_result)

    elif isinstance(pipeline_result, dict):
        results.append(pipeline_result)

    else:
        raise TypeError(
            "Unexpected MigrationPipeline result type: "
            f"{type(pipeline_result).__name__}"
        )

except Exception as exc:
    pipeline_error = f"{type(exc).__name__}: {exc}"
    traceback.print_exc()


# ============================================================
# 7. REPORT HELPER FUNCTIONS
# ============================================================

def json_safe(value):
    return json.loads(
        json.dumps(
            value,
            ensure_ascii=False,
            default=str
        )
    )


def count_items(value):
    if isinstance(value, list):
        return len(value)

    if isinstance(value, dict):
        for key in (
            "items",
            "data",
            "results",
            "records",
            "assets",
        ):
            if isinstance(value.get(key), list):
                return len(value[key])

        return len(value)

    return 0


def load_llm_result(result):
    path = result.get("llm_result_path")

    if not path or not os.path.isfile(path):
        return {}

    try:
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)

        return payload if isinstance(payload, dict) else {}

    except Exception as exc:
        return {
            "_report_read_error": str(exc)
        }

In [0]:

# ============================================================
# IDMC ENTERPRISE CUSTOMER 360
# RUN ONLY MAP100: CUSTOMER 360 ENRICHMENT
# DRY RUN - NO TARGET TABLE WRITES
# ============================================================

import json
import os
import sys
import traceback
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/Workspace/Users/"
    "lokeswaraseetaramasastry.k@tcs.com/"
    "IDMC_Databricks_LLM"
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/landing/EnterpriseCustomer360_POC_Corrected.xml"
)

OUTPUT_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark/"
    "enterprise_customer360"
)

REVIEW_DIR = os.path.join(OUTPUT_DIR, "human_reviews")

LLM_MODEL = "databricks-gpt-oss-120b"

# Explicitly select ONLY MAP100
SELECTED_MAPPING_NAME = "m_customer_360_enrichment"

# Safety settings
DRY_RUN = True
EXECUTION_APPROVED = False
ALLOW_SYNTHETIC_MERGE_KEYS = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REVIEW_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. SAFETY VALIDATION
# ------------------------------------------------------------

if not os.path.isfile(XML_PATH):
    raise FileNotFoundError(
        f"XML file not found: {XML_PATH}"
    )

if not DRY_RUN:
    raise RuntimeError(
        "This MAP100 test must run in DRY_RUN mode."
    )

if EXECUTION_APPROVED:
    raise RuntimeError(
        "Execution approval must remain False."
    )

if ALLOW_SYNTHETIC_MERGE_KEYS:
    raise RuntimeError(
        "Synthetic merge keys must remain disabled."
    )

# ------------------------------------------------------------
# 3. DATABRICKS CONTEXT
# ------------------------------------------------------------

context = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
)

WORKSPACE_URL = context.apiUrl().get()
DATABRICKS_TOKEN = context.apiToken().get()

if not WORKSPACE_URL:
    raise RuntimeError("Databricks workspace URL unavailable.")

if not DATABRICKS_TOKEN:
    raise RuntimeError("Databricks token unavailable.")

print("=" * 90)
print("IDMC MIGRATION - INDIVIDUAL MAPPING RUN")
print("=" * 90)
print("XML:", XML_PATH)
print("Selected mapping:", SELECTED_MAPPING_NAME)
print("LLM:", LLM_MODEL)
print("Dry run:", DRY_RUN)
print("Execution approved:", EXECUTION_APPROVED)
print("Synthetic merge keys:", ALLOW_SYNTHETIC_MERGE_KEYS)
print("Output:", OUTPUT_DIR)
print("=" * 90)

# ------------------------------------------------------------
# 4. IMPORT PIPELINE
# ------------------------------------------------------------

from pipeline.migration_pipeline import MigrationPipeline

# ------------------------------------------------------------
# 5. EXECUTE MAP100 ONLY
# ------------------------------------------------------------

run_started_at = datetime.now(timezone.utc)

pipeline = None
pipeline_result = None
pipeline_error = None

try:
    pipeline = MigrationPipeline(
        xml_path=XML_PATH,
        spark=spark,
        output_dir=OUTPUT_DIR,
        workspace_url=WORKSPACE_URL,
        databricks_token=DATABRICKS_TOKEN,
        llm_model=LLM_MODEL,
        selected_mapping_name=SELECTED_MAPPING_NAME,
        dry_run=DRY_RUN,
        execution_approved=EXECUTION_APPROVED,
        allow_synthetic_merge_keys=ALLOW_SYNTHETIC_MERGE_KEYS,
        verbose=True,
        human_review_dir=REVIEW_DIR,
        approved_review_id=None,
    )

    pipeline_result = pipeline.run()

except Exception as exc:
    pipeline_error = {
        "exception_type": type(exc).__name__,
        "error": str(exc),
        "traceback": traceback.format_exc(),
    }

    traceback.print_exc()

# ------------------------------------------------------------
# 6. NORMALIZE RESULT
# ------------------------------------------------------------

if isinstance(pipeline_result, list):
    mapping_results = pipeline_result

elif isinstance(pipeline_result, dict):
    mapping_results = [pipeline_result]

elif pipeline_result is None:
    mapping_results = []

else:
    mapping_results = [{
        "unexpected_result_type": type(pipeline_result).__name__
    }]

# ------------------------------------------------------------
# 7. CAPTURE METADATA
# ------------------------------------------------------------

metadata_snapshot = {}

if pipeline is not None:
    metadata_snapshot = (
        getattr(pipeline, "metadata_bundle", {}) or {}
    )

# ------------------------------------------------------------
# 8. SAVE RUN SUMMARY
# ------------------------------------------------------------

run_finished_at = datetime.now(timezone.utc)

summary = {
    "mapping": SELECTED_MAPPING_NAME,
    "xml_path": XML_PATH,
    "started_at_utc": run_started_at.isoformat(),
    "finished_at_utc": run_finished_at.isoformat(),
    "duration_seconds": (
        run_finished_at - run_started_at
    ).total_seconds(),
    "dry_run": DRY_RUN,
    "execution_approved": EXECUTION_APPROVED,
    "allow_synthetic_merge_keys": ALLOW_SYNTHETIC_MERGE_KEYS,
    "pipeline_error": pipeline_error,
    "results": mapping_results,
    "metadata_available": bool(metadata_snapshot),
}

summary_path = os.path.join(
    OUTPUT_DIR,
    "MAP100_migration_run_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as handle:
    json.dump(
        summary,
        handle,
        indent=2,
        ensure_ascii=False,
        default=str
    )

# ------------------------------------------------------------
# 9. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("MAP100 MIGRATION SUMMARY")
print("=" * 90)

print(json.dumps(
    summary,
    indent=2,
    ensure_ascii=False,
    default=str
))

print("\nSummary saved to:", summary_path)

if pipeline_error:
    print("\nMAP100 PIPELINE FAILED - CHECK TRACEBACK")

elif not mapping_results:
    print("\nMAP100 RETURNED NO RESULT - INVESTIGATION REQUIRED")

else:
    print("\nMAP100 PIPELINE FINISHED.")
    print("Review generation and validation statuses above.")

print("=" * 90)


In [0]:

# ============================================================
# MAP100 - INSPECT LATEST LLM RESULT, GENERATED CODE AND REPORT
# READ ONLY
# ============================================================

import json

BASE = (
    "dbfs:/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/enterprise_customer360"
)

FILES = {
    "LLM_RESULT": (
        BASE + "/llm_results/"
        "m_customer_360_enrichment_llm_result.json"
    ),
    "GENERATED_PYSPARK": (
        BASE + "/m_customer_360_enrichment.py"
    ),
    "LATEST_REPORT_JSON": (
        BASE + "/reports/"
        "migration_report_20261105_031408.json"
    ),
    "LATEST_REPORT_CSV": (
        BASE + "/reports/"
        "migration_report_20261105_031408.csv"
    ),
    "RUN_SUMMARY": (
        BASE + "/MAP100_migration_run_summary.json"
    )
}

def read_file(path):
    return dbutils.fs.head(path, 1000000)

# 1. LLM RESULT
print("\n" + "=" * 80)
print("1. MAP100 LLM RESULT")
print("=" * 80)

try:
    llm = json.loads(read_file(FILES["LLM_RESULT"]))

    print("Keys:", list(llm.keys()))

    for key in [
        "generation_status",
        "validation",
        "validation_history",
        "assumptions_or_gaps",
        "mapping_name"
    ]:
        if key in llm:
            print(f"\n{key}:")
            print(json.dumps(llm[key], indent=2, default=str)[:15000])

    code = llm.get("pyspark_code", "")
    print("\nLLM GENERATED CODE LENGTH:", len(code))
    print(code[:12000])

except Exception as e:
    print("LLM result inspection failed:", repr(e))

# 2. GENERATED PYSPARK FILE
print("\n" + "=" * 80)
print("2. GENERATED PYSPARK FILE")
print("=" * 80)

try:
    code = read_file(FILES["GENERATED_PYSPARK"])
    print("Code length:", len(code))
    print(code[:15000])
except Exception as e:
    print("Generated code inspection failed:", repr(e))

# 3. LATEST JSON REPORT
print("\n" + "=" * 80)
print("3. LATEST MIGRATION REPORT")
print("=" * 80)

try:
    report = json.loads(read_file(FILES["LATEST_REPORT_JSON"]))
    print(json.dumps(report, indent=2, default=str)[:25000])
except Exception as e:
    print("Report inspection failed:", repr(e))

# 4. RUN SUMMARY
print("\n" + "=" * 80)
print("4. MAP100 RUN SUMMARY")
print("=" * 80)

try:
    summary = json.loads(read_file(FILES["RUN_SUMMARY"]))
    print(json.dumps(summary, indent=2, default=str)[:15000])
except Exception as e:
    print("Summary inspection failed:", repr(e))

print("\nINSPECTION COMPLETE")


In [0]:

# ============================================================
# MAP100 ENVIRONMENT DIAGNOSTICS - READ ONLY
# No table writes, no drops, no migration execution
# ============================================================

from pyspark.sql.utils import AnalysisException

print("=" * 80)
print("MAP100 SOURCE FILE CHECK")
print("=" * 80)

source_path = (
    "/Volumes/idmc_poc/bronze/idmc_files/landing/customer.csv"
)

try:
    print("Checking:", source_path)
    print(dbutils.fs.ls(
        "/Volumes/idmc_poc/bronze/idmc_files/landing/"
    ))

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(source_path)
    )

    print("CUSTOMER SOURCE SCHEMA:")
    df.printSchema()
    print("CUSTOMER SAMPLE:")
    df.show(5, truncate=False)

except Exception as e:
    print("CUSTOMER SOURCE ERROR:")
    print(type(e).__name__)
    print(str(e)[:5000])

print("\n" + "=" * 80)
print("MAP100 TABLE CHECK")
print("=" * 80)

tables = [
    "idmc_poc.sales.orders",
    "idmc_poc.reference.region",
    "idmc_poc.curated.customer_gold",
    "idmc_poc.curated.customer_standard",
    "idmc_poc.curated.customer_low_value",
    "idmc_poc.quality.customer_360_rejects",
    "idmc_poc.quality.notification_outbox"
]

for table_name in tables:
    print("\nTABLE:", table_name)

    try:
        table_df = spark.table(table_name)
        print("EXISTS: YES")
        print("COLUMNS:", table_df.columns)
        table_df.printSchema()

    except Exception as e:
        print("EXISTS: NO / INACCESSIBLE")
        print(type(e).__name__, str(e)[:1500])

print("\nDIAGNOSTICS COMPLETED")


In [0]:
# ============================================================
# FIX MAP100 CUSTOMER CSV SOURCE
# ============================================================

landing_path = "/Volumes/idmc_poc/bronze/idmc_files/landing"
customer_path = f"{landing_path}/customer.csv"

customer_csv = """customer_id,first_name,last_name,email,region_code,updated_ts
C001,Amit,Sharma,amit@example.com,IN,2026-10-01T10:00:00.000Z
C002,Rahul,Kumar,rahul@example.com,US,2026-10-02T11:00:00.000Z
C003,Priya,Reddy,priya@example.com,UK,2026-10-03T12:00:00.000Z
"""

# Replace existing file
dbutils.fs.put(
    customer_path,
    customer_csv,
    overwrite=True
)

print("CUSTOMER CSV UPDATED")

# Read and validate
customer_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(customer_path)
)

print("Rows:", customer_df.count())
customer_df.show(truncate=False)

# Validate customer ID overlap
orders_df = spark.table("idmc_poc.sales.orders")

print("CUSTOMER ID JOIN VALIDATION")

customer_df.select("customer_id").distinct().join(
    orders_df.select("customer_id").distinct(),
    "customer_id",
    "inner"
).show(truncate=False)

# Validate region codes
region_df = spark.table("idmc_poc.reference.region")

print("REGION LOOKUP VALIDATION")

customer_df.select("region_code").distinct().join(
    region_df.select("region_code").distinct(),
    "region_code",
    "inner"
).show(truncate=False)

print("MAP100 SOURCE DATA FIX COMPLETED")

In [0]:
# ============================================================
# ENTERPRISE CUSTOMER 360
# COMPLETE SOURCE READINESS VALIDATION
# 20 TABLE SOURCES + 2 FILE SOURCES
# ============================================================

from pyspark.sql import functions as F

CATALOG = "idmc_poc"

TABLE_SOURCES = [
    "idmc_poc.sales.orders",
    "idmc_poc.reference.region",
    "idmc_poc.silver.customer",
    "idmc_poc.bronze.order_status",
    "idmc_poc.bronze.products",
    "idmc_poc.bronze.inventory",
    "idmc_poc.bronze.product_category",
    "idmc_poc.bronze.customer_consent",
    "idmc_poc.bronze.privacy_policy",
    "idmc_poc.bronze.transaction",
    "idmc_poc.bronze.chargebacks",
    "idmc_poc.bronze.device_trust",
    "idmc_poc.bronze.web_events",
    "idmc_poc.bronze.campaigns",
    "idmc_poc.bronze.invoice",
    "idmc_poc.bronze.credit_note",
    "idmc_poc.bronze.purchase_orders",
    "idmc_poc.bronze.suppliers",
    "idmc_poc.bronze.support_cases",
    "idmc_poc.bronze.sla_policy"
]

FILE_SOURCES = {
    "MAP100_customer": (
        "/Volumes/idmc_poc/bronze/idmc_files/landing/customer.csv",
        [
            "customer_id", "first_name", "last_name",
            "email", "region_code", "updated_ts"
        ]
    ),
    "MAP101_orders": (
        "/Volumes/idmc_poc/bronze/idmc_files/landing/orders_delta.csv",
        [
            "order_id", "customer_id", "order_status",
            "amount", "updated_ts"
        ]
    )
}

results = []

print("=" * 90)
print("PHASE 1: TABLE SOURCE VALIDATION")
print("=" * 90)

for table_name in TABLE_SOURCES:

    try:
        df = spark.table(table_name)

        row_count = df.count()
        columns = df.columns

        results.append({
            "source": table_name,
            "type": "TABLE",
            "status": "PASS",
            "rows": row_count,
            "details": f"{len(columns)} columns"
        })

        print(
            f"PASS | {table_name} | "
            f"Rows={row_count} | Columns={len(columns)}"
        )

    except Exception as e:

        results.append({
            "source": table_name,
            "type": "TABLE",
            "status": "FAIL",
            "rows": 0,
            "details": str(e)[:300]
        })

        print(f"FAIL | {table_name} | {str(e)[:300]}")


print("\n" + "=" * 90)
print("PHASE 2: FILE SOURCE VALIDATION")
print("=" * 90)

for source_name, config in FILE_SOURCES.items():

    path, expected_columns = config

    try:
        df = (
            spark.read
            .option("header", "true")
            .option("mode", "FAILFAST")
            .csv(path)
        )

        actual_columns = df.columns
        row_count = df.count()

        missing = [
            col for col in expected_columns
            if col not in actual_columns
        ]

        extra = [
            col for col in actual_columns
            if col not in expected_columns
        ]

        if missing:
            status = "FAIL"
            details = f"Missing columns: {missing}"

        elif extra:
            status = "WARNING"
            details = f"Unexpected columns: {extra}"

        else:
            status = "PASS"
            details = f"{len(actual_columns)} expected columns"

        results.append({
            "source": source_name,
            "type": "FILE",
            "status": status,
            "rows": row_count,
            "details": details
        })

        print(
            f"{status} | {source_name} | "
            f"Rows={row_count} | {details}"
        )

    except Exception as e:

        results.append({
            "source": source_name,
            "type": "FILE",
            "status": "FAIL",
            "rows": 0,
            "details": str(e)[:300]
        })

        print(f"FAIL | {source_name} | {str(e)[:300]}")


print("\n" + "=" * 90)
print("PHASE 3: JOIN COMPATIBILITY")
print("=" * 90)

# MAP100: customer -> sales orders
customer_df = spark.read.option("header", True).csv(
    FILE_SOURCES["MAP100_customer"][0]
)

orders_df = spark.table("idmc_poc.sales.orders")

customer_matches = (
    customer_df.select("customer_id").distinct()
    .join(
        orders_df.select("customer_id").distinct(),
        "customer_id",
        "inner"
    )
    .count()
)

print(f"MAP100 customer/order matches: {customer_matches}")

# MAP100: customer -> region
region_df = spark.table("idmc_poc.reference.region")

region_matches = (
    customer_df.select("region_code").distinct()
    .join(
        region_df.select("region_code").distinct(),
        "region_code",
        "inner"
    )
    .count()
)

print(f"MAP100 customer/region matches: {region_matches}")

# MAP101: orders -> silver customer
orders_file_df = spark.read.option("header", True).csv(
    FILE_SOURCES["MAP101_orders"][0]
)

silver_customer_df = spark.table("idmc_poc.silver.customer")

customer_matches_map101 = (
    orders_file_df.select("customer_id").distinct()
    .join(
        silver_customer_df.select("customer_id").distinct(),
        "customer_id",
        "inner"
    )
    .count()
)

print(
    f"MAP101 orders/customer matches: "
    f"{customer_matches_map101}"
)

print("\n" + "=" * 90)
print("FINAL SOURCE READINESS SUMMARY")
print("=" * 90)

summary_df = spark.createDataFrame(results)

summary_df.show(30, truncate=False)

summary_df.groupBy("type", "status").count().show()

failed = [
    r for r in results
    if r["status"] == "FAIL"
]

warnings = [
    r for r in results
    if r["status"] == "WARNING"
]

print("Total sources:", len(results))
print("Failed sources:", len(failed))
print("Warnings:", len(warnings))

if not failed and not warnings:
    print("\nALL 22 SOURCE CHECKS PASSED")
else:
    print("\nSOURCE VALIDATION NEEDS ATTENTION")

In [0]:

# ============================================================
# RUN IDMC MIGRATION ENGINE - FIRST TWO MAPPINGS
# CUSTOMER 360 + ORDERS SCD2
# ============================================================

import os
import sys
import json
import traceback
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/Workspace/Users/"
    "lokeswaraseetaramasastry.k@tcs.com/"
    "IDMC_Databricks_LLM"
)

XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "IDMC_Advanced_Complex_Metadata_Repository.xml"
)

OUTPUT_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/enterprise_customer360"
)

REVIEW_DIR = os.path.join(OUTPUT_DIR, "human_reviews")

MAPPINGS_TO_RUN = [
    "m_customer_360_enrichment",
    "m_orders_enrichment_scd2",
]

LLM_MODEL = "databricks-gpt-oss-120b"

# SAFETY CONTROLS
DRY_RUN = True
EXECUTION_APPROVED = False
ALLOW_SYNTHETIC_MERGE_KEYS = False

# ------------------------------------------------------------
# 2. IMPORT PIPELINE
# ------------------------------------------------------------

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.migration_pipeline import MigrationPipeline

# ------------------------------------------------------------
# 3. VALIDATE INPUTS
# ------------------------------------------------------------

if not os.path.exists(XML_PATH):
    raise FileNotFoundError(
        "Repository XML not found: " + XML_PATH
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REVIEW_DIR, exist_ok=True)

# ------------------------------------------------------------
# 4. GET DATABRICKS CREDENTIALS
# ------------------------------------------------------------

context = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
)

WORKSPACE_URL = context.apiUrl().get()
DATABRICKS_TOKEN = context.apiToken().get()

# ------------------------------------------------------------
# 5. INITIALIZE MIGRATION ENGINE
# ------------------------------------------------------------

print("=" * 80)
print("INITIALIZING IDMC MIGRATION ENGINE")
print("=" * 80)

pipeline = MigrationPipeline(
    xml_path=XML_PATH,
    spark=spark,
    output_dir=OUTPUT_DIR,
    workspace_url=WORKSPACE_URL,
    databricks_token=DATABRICKS_TOKEN,
    llm_model=LLM_MODEL,
    selected_mapping_name=None,
    dry_run=DRY_RUN,
    execution_approved=EXECUTION_APPROVED,
    allow_synthetic_merge_keys=ALLOW_SYNTHETIC_MERGE_KEYS,
    verbose=True,
    human_review_dir=REVIEW_DIR,
    approved_review_id=None,
)

print("Pipeline initialized.")
print("Dry run:", pipeline.dry_run)
print("Execution approved:", pipeline.execution_approved)

# ------------------------------------------------------------
# 6. READ AND PARSE REPOSITORY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("READING IDMC XML")
print("=" * 80)

root = pipeline.xml_reader.read()
repository = pipeline.repository_parser.parse(root)

folders = repository.get("folders", [])

all_mappings = []

for folder in folders:
    folder_name = folder.get("name", "UNKNOWN")

    for mapping in folder.get("mappings", []):
        if isinstance(mapping, dict):
            mapping_copy = dict(mapping)
            mapping_copy["_debug_folder"] = folder_name
            all_mappings.append(mapping_copy)

print("Repository:", repository.get("name"))
print("Folders discovered:", len(folders))
print("Mappings discovered:", len(all_mappings))

# ------------------------------------------------------------
# 7. FIND THE EXACT TWO MAPPINGS
# ------------------------------------------------------------

mapping_lookup = {}

for mapping in all_mappings:
    name = mapping.get("name") or mapping.get("mapping_name")

    if name:
        mapping_lookup[name] = mapping

missing = [
    name for name in MAPPINGS_TO_RUN
    if name not in mapping_lookup
]

if missing:
    print("\nAvailable mapping names:")
    for name in sorted(mapping_lookup.keys()):
        print(" -", name)

    raise ValueError(
        "Required mappings missing from XML: " + ", ".join(missing)
    )

print("\nMappings selected:")

for name in MAPPINGS_TO_RUN:
    print(" -", name)

# ------------------------------------------------------------
# 8. PARSE SPECIALIZED METADATA
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PARSING SPECIALIZED METADATA")
print("=" * 80)

pipeline.parse_specialized_metadata(
    root=root,
    repository=repository
)

print("Specialized metadata parsing completed.")

# ------------------------------------------------------------
# 9. PROCESS BOTH MAPPINGS THROUGH THE ENGINE
# ------------------------------------------------------------

debug_results = []

for index, mapping_name in enumerate(MAPPINGS_TO_RUN, start=1):

    print("\n" + "#" * 80)
    print("RUNNING MAPPING", index, "OF", len(MAPPINGS_TO_RUN))
    print("MAPPING:", mapping_name)
    print("#" * 80)

    mapping = mapping_lookup[mapping_name]

    try:
        result = pipeline.process_mapping(mapping)

        if not isinstance(result, dict):
            result = {
                "mapping_name": mapping_name,
                "success": False,
                "stage": "UNEXPECTED_RETURN_TYPE",
                "raw_result": str(result),
            }

        result.setdefault("mapping_name", mapping_name)
        debug_results.append(result)

    except Exception as error:

        error_trace = traceback.format_exc()

        print("\nMAPPING FAILED:", repr(error))
        print(error_trace)

        debug_results.append({
            "mapping_name": mapping_name,
            "success": False,
            "stage": "ENGINE_EXCEPTION",
            "error": repr(error),
            "traceback": error_trace,
        })

# ------------------------------------------------------------
# 10. DISPLAY ENGINE RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TWO-MAPPING ENGINE RESULTS")
print("=" * 80)

for result in debug_results:

    mapping_name = result.get("mapping_name", "UNKNOWN")
    llm_result = result.get("llm_result") or {}
    validation = result.get("validation") or {}
    generator_validation = llm_result.get("validation") or {}

    print("\nMapping:", mapping_name)
    print("Stage:", result.get("stage"))
    print("Success:", result.get("success"))
    print("Generation status:", llm_result.get("generation_status"))
    print("Final validation:", validation.get("valid"))
    print("Generated PySpark:", result.get("generated_pyspark"))
    print("LLM result:", result.get("llm_result_path"))

    print("\nFinal validation errors:")
    for error in validation.get("errors", []):
        print(" -", error)

    print("\nGenerator validation errors:")
    for error in generator_validation.get("errors", []):
        print(" -", error)

    print("\nBlocking issues:")
    for issue in result.get("blocking_issues", []):
        print(" -", issue)

# ------------------------------------------------------------
# 11. SAVE CONSOLIDATED DEBUG REPORT
# ------------------------------------------------------------

debug_summary = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "xml_path": XML_PATH,
    "selected_mappings": MAPPINGS_TO_RUN,
    "dry_run": DRY_RUN,
    "execution_approved": EXECUTION_APPROVED,
    "allow_synthetic_merge_keys": ALLOW_SYNTHETIC_MERGE_KEYS,
    "results": debug_results,
}

summary_path = os.path.join(
    OUTPUT_DIR,
    "two_mapping_engine_summary.json"
)

try:
    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(
            debug_summary,
            file,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    print("\nSummary saved:", summary_path)

except Exception as error:
    print("\nSummary save failed:", repr(error))

# ------------------------------------------------------------
# 12. FINAL STATUS
# ------------------------------------------------------------

successful = sum(
    1 for result in debug_results
    if result.get("success") is True
)

print("\n" + "=" * 80)
print("MIGRATION ENGINE RUN COMPLETED")
print("=" * 80)
print("Mappings requested:", len(MAPPINGS_TO_RUN))
print("Mappings returned:", len(debug_results))
print("Successful mappings:", successful)
print("Failed mappings:", len(debug_results) - successful)
print("Dry run:", DRY_RUN)
print("Generated PySpark executed: NO")
print("Target Delta writes performed: NO")
print("=" * 80)


In [0]:
PYSPARK_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/enterprise_customer360/"
    "m_customer_360_enrichment.py"
)

print(dbutils.fs.head(PYSPARK_PATH, 50000))

In [0]:
LLM_RESULT_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/enterprise_customer360/"
    "llm_results/m_customer_360_enrichment_llm_result.json"
)

print(dbutils.fs.head(LLM_RESULT_PATH, 100000))

In [0]:
LLM_RESULT_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/enterprise_customer360/"
    "llm_results/m_customer_360_enrichment_llm_result.json"
)

print(dbutils.fs.head(LLM_RESULT_PATH, 100000))

In [0]:

# ============================================================
# IDMC -> DATABRICKS
# RUN: m_orders_enrichment_scd2
# DRY RUN ONLY - NO TARGET WRITES
# ============================================================

import os
import sys
import json
import traceback
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/Workspace/Users/"
    "lokeswaraseetaramasastry.k@tcs.com/"
    "IDMC_Databricks_LLM"
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "IDMC_Advanced_Complex_Metadata_Repository.xml"
)

OUTPUT_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/enterprise_customer360"
)

REVIEW_DIR = os.path.join(OUTPUT_DIR, "human_reviews")

MAPPING_NAME = "m_orders_enrichment_scd2"
LLM_MODEL = "databricks-gpt-oss-120b"

DRY_RUN = True
EXECUTION_APPROVED = False
ALLOW_SYNTHETIC_MERGE_KEYS = False

ORDERS_SOURCE_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "landing/orders_delta.csv"
)

# ------------------------------------------------------------
# 2. SAFETY AND PATH CHECKS
# ------------------------------------------------------------

if not DRY_RUN or EXECUTION_APPROVED:
    raise RuntimeError(
        "This runner is restricted to dry-run mode."
    )

if not os.path.isdir(PROJECT_ROOT):
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_ROOT}"
    )

if not os.path.isfile(XML_PATH):
    raise FileNotFoundError(
        f"IDMC XML not found: {XML_PATH}"
    )

if not os.path.exists(ORDERS_SOURCE_PATH):
    raise FileNotFoundError(
        f"Orders CSV source not found: {ORDERS_SOURCE_PATH}"
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REVIEW_DIR, exist_ok=True)

print("=" * 80)
print("IDMC MIGRATION - ORDERS MAPPING")
print("=" * 80)
print("Mapping:", MAPPING_NAME)
print("XML:", XML_PATH)
print("Source:", ORDERS_SOURCE_PATH)
print("Dry run:", DRY_RUN)
print("Execution approved:", EXECUTION_APPROVED)

# ------------------------------------------------------------
# 3. VERIFY THE ACTUAL CSV SOURCE
# ------------------------------------------------------------

orders_check = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ORDERS_SOURCE_PATH)
)

EXPECTED_COLUMNS = [
    "order_id",
    "customer_id",
    "order_status",
    "amount",
    "updated_ts"
]

if orders_check.columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Orders CSV columns do not match the verified source contract.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual: {orders_check.columns}"
    )

print("\nSource schema verified:")
orders_check.printSchema()

print("Source sample:")
display(orders_check.limit(5))

# ------------------------------------------------------------
# 4. GET DATABRICKS WORKSPACE CONTEXT
# ------------------------------------------------------------

context = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
)

WORKSPACE_URL = context.apiUrl().get()
DATABRICKS_TOKEN = context.apiToken().get()

if not WORKSPACE_URL or not DATABRICKS_TOKEN:
    raise RuntimeError(
        "Databricks workspace URL or notebook token unavailable."
    )

# ------------------------------------------------------------
# 5. IMPORT MIGRATION PIPELINE
# ------------------------------------------------------------

from pipeline.migration_pipeline import MigrationPipeline

# ------------------------------------------------------------
# 6. RUN ONLY THE SELECTED MAPPING
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STARTING MAPPING:", MAPPING_NAME)
print("=" * 80)

started_at = datetime.now(timezone.utc)

pipeline = MigrationPipeline(
    xml_path=XML_PATH,
    spark=spark,
    output_dir=OUTPUT_DIR,
    workspace_url=WORKSPACE_URL,
    databricks_token=DATABRICKS_TOKEN,
    llm_model=LLM_MODEL,
    selected_mapping_name=MAPPING_NAME,
    dry_run=True,
    execution_approved=False,
    allow_synthetic_merge_keys=False,
    verbose=True,
    human_review_dir=REVIEW_DIR,
    approved_review_id=None,
)

result = pipeline.run()

finished_at = datetime.now(timezone.utc)

# ------------------------------------------------------------
# 7. DISPLAY RESULT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MAPPING RUN COMPLETED")
print("=" * 80)

print("Started:", started_at.isoformat())
print("Finished:", finished_at.isoformat())
print("Duration:", (finished_at - started_at).total_seconds(), "seconds")

if isinstance(result, dict):
    print(json.dumps(result, indent=2, default=str))
elif isinstance(result, list):
    print(json.dumps(result, indent=2, default=str))
else:
    print("Pipeline result:", result)

print("\nGenerated artifacts directory:")
print(OUTPUT_DIR)

print("\nExecution mode: DRY RUN")
print("No target-table writes were approved.")


In [0]:
catalog = "idmc_poc"

for schema in spark.sql(f"SHOW SCHEMAS IN {catalog}").collect():
    schema_name = schema["databaseName"]

    print(f"\n{'=' * 60}")
    print(f"SCHEMA: {schema_name}")
    print("=" * 60)

    try:
        tables = spark.sql(
            f"SHOW TABLES IN {catalog}.{schema_name}"
        ).collect()

        for table in tables:
            print(table["tableName"])

    except Exception as e:
        print(f"Could not list tables: {e}")

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/idmc_poc/bronze/idmc_files/landing/"
    )
)